# Introduction

This notebook contains all code used for Samantha Anwar's Master's thesis at Columbia University. 

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import geopandas as gpd
import requests
import os
import glob
import datetime
import pytz
from scipy.optimize import minimize
import json
from shapely.geometry import shape, LineString, Point
import linearmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
import patsy
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

# 1. County background

Create a dataframe of your adjustment variables to make appropriate comparisons across counties. Look at GDP, population, income, and snowfall per county. Additionally, add variables for whether a county is serviced by ERCOT or not.

## 1.1 FIPS codes and geometries

In [ ]:
def load_fips_shapes():

    fips_shapes = gpd.read_file('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json')
    statefips = pd.read_excel('data/state_fips.xlsx')

    statefips.STATEFP = statefips.STATEFP.astype('str')
    statefips.STATEFP = ['0' + state if len(state) < 2 else state for state in statefips.STATEFP]
    
    fips_shapes = fips_shapes.rename(columns = {'STATE': 'STATEFP', 'COUNTY':'COUNTYFP'})
    finalfips = fips_shapes.merge(statefips, on = 'STATEFP')

    finalfips = finalfips.rename(columns = {'NAME': 'county', 'STATE': 'state_abbr', 
                                            'id': 'fips_code', 'STATE_NAME': 'state',
                                            'CENSUSAREA': 'census_area'})
    
    finalfips['county'] = finalfips['county'].str.title()
    
    return finalfips[['fips_code', 'county', 'state_abbr', 'state', 'geometry', 'census_area', ]]

fips_shapes = load_fips_shapes()

## 1.2 Snowfall

Data source: https://www.ncei.noaa.gov/access/monitoring/daily-snow/

In [ ]:
snowfall = pd.read_csv('data/snowfall.csv')

# filter to Texas and Oklahoma; replace missing values
snowfall = (snowfall
            .loc[snowfall.State.isin(['TX', 'OK', 'LA', 'NM'])]
            .replace('M', np.nan)
            .replace('T', np.nan))
date_cols = snowfall.columns[-28:]
id_cols = snowfall.columns[:-28]

# transform df from wide to long format on date
snowfall = (snowfall
            .melt(
                id_vars = id_cols, 
                value_vars = date_cols, 
                var_name = 'Date', 
                value_name='Snowfall')
            .rename(columns={
                'State': 'state_abbr', 
                'County': 'county',
                'Snowfall': 'snowfall',
                'Date': 'date'}))

snowfall.county = snowfall.county.str.title()

snowfall.date = snowfall.date + '-2021'
snowfall.date = pd.to_datetime(snowfall.date, format = '%d-%b-%Y')


snowfall = snowfall.loc[(snowfall.date >= '2021-02-10') & (snowfall.date <= '2021-02-20')].reset_index(drop=True)
snowfall.snowfall = snowfall.snowfall.astype('float')

fips_ids = fips_shapes[['state_abbr', 'state', 'county', 'fips_code']]
snowfall = (snowfall
            .merge(fips_ids, on = ['state_abbr', 'county'])
            .drop(columns = ['GHCN_ID', 'City/Station Name', 'Elev', 'Lat', 'Lon']))

# find daily max per county
dailymax_snowfall = snowfall.groupby(by = ['state_abbr', 'county', 'date', 'state', 'fips_code']).max().reset_index()
dailymax_snowfall.head(3)

In [ ]:
# Load US counties GeoJSON
geojson_url = 'https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json'
counties = requests.get(geojson_url).json()

# Load US states GeoJSON
states_url = 'https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json'
states_geojson = requests.get(states_url).json()

states_geojson['features'] = [
    feature for feature in states_geojson['features']
    if feature['properties']['name'] in ['Texas', 'Oklahoma', 'Louisiana', 'New Mexico']
]

### 1.2.1 Total snowfall for storm duration

Remove time series element so that we have a singular snowfall number for each county for future DiD analysis. We'll take a sum by county rather than average so that 0 snowfall days do not skew the severity. 

In [ ]:
# df will be main dataframe with these control variables
df = fips_shapes.copy().drop(columns='geometry')

snow = dailymax_snowfall[['state_abbr', 'county', 'state', 'fips_code', 'snowfall']]
snow = snow.groupby(by=['state_abbr', 'county', 'state', 'fips_code']).sum().reset_index()

df = df.merge(snow, on = ['state_abbr', 'county', 'state', 'fips_code'])
df = df[(df.state_abbr != 'NM')]

df.head(3)

In [ ]:
df.groupby('state')['snowfall'].mean()

In [ ]:
fig = go.Figure(data=go.Choropleth(
    locations=df['fips_code'],
    geojson=counties,
    z=df['snowfall'],
    colorscale='Blues',
    zmin=0,
    zmax=df['snowfall'].max(),
    marker_line_width=0,
    hoverinfo='location+z',
    customdata=df[['county']],
    hovertemplate='<b>%{customdata[0]} County</b><br><br>Snowfall: %{z} in<extra></extra>',
    colorbar=dict(
        title='Snowfall (in)',
        x=0.8,         # Move closer to plot (1 is far right)
        len=0.4,        # Shorten height
        y=0.8,          # Center vertically
        thickness=20,   # Narrower bar
        xpad=10         # Padding between plot and bar
    )))

fig.add_trace(go.Choropleth(
    geojson=states_geojson,
    locations=['Texas', 'Oklahoma', 'Louisiana'],
    featureidkey="properties.name",
    z=[0, 0, 0],  # Dummy data
    showscale=False,
    marker_line_color="black",
    marker_line_width=1,
    colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],
    hoverinfo='skip'
))

fig.update_layout(
    height=400, width=600,
    title=dict(
        text='Uri Snowfall Totals by County',
        x=0, y=0.975),
    margin=dict(t=40, b=0, l=0, r=0),
    dragmode=False, font_family='Arial')

fig.add_annotation(
    text='February 10 - 20, 2021',
    x=0, y=1.03,
    xref='paper', yref='paper',
    showarrow=False,
    font=dict(size=14)
)

fig.update_geos(
    visible=False,
    center={"lat": 31.75, "lon": -97.0},
    lonaxis_range=[-107.65, -88],
    lataxis_range=[25.9, 38.2])

fig.update_layout(dragmode=False)
fig.show()
fig.write_html('html plots/snowfall.html')

## 1.3 Labeling ERCOT counties

In [ ]:
non_ercot = [
    'DALLAM', 'SHERMAN', 'HANSFORD', 'OCHILTREE', 'LIPSCOMB', 'HARTLEY', 'MOORE', 'HUTCHINSON', 'HEMPHILL',
    'BAILEY', 'LAMB', 'COCHRAN', 'HOCKLEY', 'YOAKUM', 'TERRY', 'GAINES', 'EL PASO', 'HUDSPETH',
    'BOWIE', 'MORRIS', 'CASS', 'CAMP', 'UPSHUR', 'MARION', 'HARRISON', 'GREGG', 'PANOLA', 'SHELBY', 'SAN AUGUSTINE',
    'SABINE', 'TRINITY', 'POLK', 'SAN JACINTO', 'TYLER', 'JASPER', 'NEWTON', 'LIBERTY', 'HARDIN', 'ORANGE', 'JEFFERSON'
]

non_ercot = [county.title() for county in non_ercot]

ercot = []
for index, row in df.iterrows():
    if row['state_abbr'] == 'TX':
        if row.county in non_ercot:
            ercot.append(0)
        else:
            ercot.append(1)
    else:
        ercot.append(0)

df['ercot'] = ercot
df.head(3)

In [ ]:
ercot_labels = df[['fips_code', 'county', 'state_abbr', 'state', 'ercot']].drop_duplicates()

## 1.3 Data from the Bureau of Economic Analysis
### Population, median income, GDP per county

* https://www.bea.gov/data/gdp/gdp-county-metro-and-other-areas
* https://www.bea.gov/data/income-saving/personal-income-county-metro-and-other-areas


In [ ]:
gdp = pd.read_csv('data/bea_gdp.csv', skiprows=3).rename(columns={'GeoFips': 'fips_code'})

gdp['county'] = [county.split(', ')[0] for county in gdp.GeoName]
gdp['state_abbr'] = [county.split(', ')[1] for county in gdp.GeoName]

gdp = gdp.loc[(gdp.LineCode==1)][['fips_code', 'county', 'state_abbr', '2021']].rename(columns={'2021':'real_gdp'})
gdp.fips_code = gdp.fips_code.astype('str')

In [ ]:
bea = (
    pd.read_csv('data/bea.csv', skiprows=3)
      .rename(columns={'GeoFips': 'fips_code'})
      .drop(columns='LineCode')
      .reset_index(drop=True)
)

bea['county'] = [county.split(', ')[0] for county in bea.GeoName]
bea['state_abbr'] = [county.split(', ')[1] for county in bea.GeoName]
bea.fips_code = bea.fips_code.astype('str')

bea = (
    bea
       .pivot(index = ['fips_code', 'GeoName', 'county', 'state_abbr'], 
              columns = 'Description', values='2021')
       .reset_index()
       .drop(columns='GeoName')
       .rename(columns = {'Per capita personal income (dollars) 2': 'per_capita_income',
                          'Personal income (thousands of dollars)': 'personal_income_thou',
                          'Population (persons) 1': 'population'})
       .merge(gdp, on = ['fips_code', 'county', 'state_abbr'])
)

In [ ]:
# merge with df with snowfall
df = df.merge(bea, on = ['fips_code', 'county', 'state_abbr'])

#add log transforms 
df['log_income'] = np.log(df['personal_income_thou'])
df['log_gdp'] = np.log(df['real_gdp'])
df['log_pop'] = np.log(df['population'])

df.head(3)

# 2. Energy market information

## 2.1 ERCOT Energy generation and forecasted demand during the storm

In [ ]:
demand = pd.read_excel('data/energy_gen.xlsx', sheet_name = 'Chart 1 Data').rename(columns={'Gas':'Natural Gas'})
demand['Date'] = pd.to_datetime(demand['Date']).dt.round('h')

demand.head(3)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
sources = ['Natural Gas', 'Coal', 'Nuclear', 'Wind', 'Solar', 'Other']
colors = ['#03A9F4', '#8BC34A', '#FFC107', '#FF5722', '#9C27B0', '#607D8B']

# Cumulative stacking
cumulative = demand[sources[0]].copy()

# Cumulative stacking
cumulative = demand[sources[0]].copy()
fig.add_trace(go.Scatter(
    x=demand['Date'],
    y=cumulative,
    customdata=demand[[sources[0]]],
    mode='lines',
    name=sources[0],
    line=dict(width=0.5, color=colors[0]),
    fill='tozeroy',
    hovertemplate=f"{sources[0]}: %{{customdata[0]:.2f}} MWh<extra></extra>"
))

for i in range(1, len(sources)):
    cumulative += demand[sources[i]]
    fig.add_trace(go.Scatter(
        x=demand['Date'],
        y=cumulative,
        customdata=demand[[sources[i]]],
        mode='lines',
        name=sources[i],
        line=dict(width=0.5, color=colors[i]),
        fill='tonexty',
        hovertemplate=f"{sources[i]}: %{{customdata[0]:.2f}} MWh<extra></extra>"
    ))

# Add demand forecast line
fig.add_trace(go.Scatter(
    x=demand['Date'], y=demand['Demand forecast'],
    mode='lines',
    name='Demand Forecast',
    line=dict(color='black', width=2, dash='dot'),
    hovertemplate=f"Demand: %{{y:.2f}} MWh<extra></extra>"
))

# Add customers without power on secondary y-axis
fig.add_trace(go.Scatter(
    x=demand['Date'], y=demand['Outages'],
    mode='lines',
    name='Outages',
    line=dict(width=3, color='black'),
    yaxis='y2',
    hovertemplate=f"Outages: %{{y:.2f}} million<extra></extra>"
))

fig.update_layout(
    title='Electricity Generation with Demand Forecast',
    yaxis_title='Megawatthours',
    font=dict(family='Arial', size=12),
    height=400,
    width=700,
    plot_bgcolor='white',
    paper_bgcolor='white',
    yaxis=dict(showline=True, showgrid=True, zeroline=False),
    yaxis2=dict(
        title='Customers w/o Power (millions)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    legend=dict(yanchor='bottom', y=0.6, xanchor='center', x=0.85),
    hovermode='x unified'
)

fig.show()


In [ ]:
# fig.write_html('html plots/energy_generation.html')

## 2.2 Comparing generation and demand forecast across ISOs

* https://portal.spp.org/groups/operations
* https://www.eia.gov/electricity/wholesalemarkets/data.php?rto=spp

In [ ]:
ercot_demand = pd.read_csv('data/ercot_demand.csv')
ercot_demand['datetime'] = pd.to_datetime(ercot_demand['Local Date']) + pd.to_timedelta(ercot_demand['Hour Number'], unit='h')
ercot_demand['Available Power'] = ercot_demand['Total Interchange (MWh)'] * -1 + ercot_demand['Net Generation (MWh)']
ercot_demand.head(3)

In [ ]:
spp_demand = pd.read_csv('data/spp_demand.csv')
spp_demand['datetime'] = pd.to_datetime(spp_demand['Local Date']) + pd.to_timedelta(spp_demand['Hour Number'], unit='h')
spp_demand['Available Power'] = spp_demand['Total Interchange (MWh)'] * -1 + spp_demand['Net Generation (MWh)']
spp_demand.head(3)

## 2.3 Locational marginal pricing
### Comparing MISO and ERCOT

https://www.eia.gov/electricity/wholesalemarkets/index.php

In [ ]:
def clean_file(df, cols):
    df = df[cols].copy()
    df['Local Date'] = pd.to_datetime(df['Local Date'], format = '%Y-%m-%d')
    df = df.loc[(df['Local Date'] >= '2021-02-01') & (df['Local Date'] <= '2021-02-28')]
    df = df.groupby(by = ['Local Date', 'Hour Number']).mean().reset_index()
    df = df.groupby(by = ['Local Date']).mean().reset_index()
    df = df.drop(columns = 'Hour Number')
    return df

In [ ]:
miso_file = pd.read_csv('data/MISO.csv', skiprows = 3)

miso_columns = ['Local Date', 'Hour Number', 'Arkansas Hub LMP', 'Illinois Hub LMP',
                'Indiana Hub LMP', 'Louisiana Hub LMP', 'Michigan Hub LMP',
                'Minnesota Hub LMP', 'Mississippi Hub LMP', 'Texas Hub LMP']

miso = clean_file(miso_file, miso_columns)

# compute MISO average to compare to ERCOT
miso['MISO average LMP'] = miso[miso.columns[1:].to_list()].mean(axis=1)
miso_avg = miso[['Local Date', 'MISO average LMP', 'Texas Hub LMP']]

ercot_file = pd.read_csv('data/ERCOT.csv', skiprows = 3)

ercot_columns = ['Local Date', 'Hour Number', 'Bus average LMP', 'Houston LMP', 
                 'Hub average LMP', 'North LMP', 'Panhandle LMP', 
                 'South LMP', 'West LMP']

ercot = clean_file(ercot_file, ercot_columns)
ercot = ercot.rename(columns = {'Hub average LMP': 'ERCOT average LMP'})

ercot_avg = ercot[['Local Date', 'ERCOT average LMP']]

In [ ]:
ercot_miso = ercot_avg.merge(miso_avg, on = 'Local Date')
ercot_miso = ercot_miso.rename(columns = {'Texas Hub LMP': 'MISO TX Hub LMP'})

ercot_miso.head(3)

In [ ]:
# plot LMPs

color = ['#03A9F4', '#8BC34A', '#FFC107', '#FF5722', '#9C27B0', '#607D8B']

fig = go.Figure()

fig.add_trace(go.Scatter(x = ercot_miso['Local Date'], 
                         y=ercot_miso['ERCOT average LMP'], 
                         name='ERCOT',
                         line=dict(color=color[0], width=2, 
                                   shape = 'spline', smoothing=0.8),
                         hovertemplate = f"ERCOT: $%{{y:.2f}}<extra></extra>"))

fig.add_trace(go.Scatter(x = ercot_miso['Local Date'], 
                         y=ercot_miso['MISO average LMP'], 
                         name='MISO',
                         line=dict(color=color[1], width=2, 
                                   shape = 'spline', smoothing=0.8),
                         hovertemplate = f"MISO: $%{{y:.2f}}<extra></extra>"))

fig.add_trace(go.Scatter(x = ercot_miso['Local Date'], 
                         y=ercot_miso['MISO TX Hub LMP'], 
                         name='MISO TX Hub',
                         line=dict(color=color[2], width=2, 
                                   shape = 'spline', smoothing=0.8),
                         hovertemplate = f"MISO TX Hub: $%{{y:.2f}}<extra></extra>"))

fig.update_layout(yaxis_title = 'Avg Locational Marginal Pricing ($/MW)',
                  height = 400, width = 700, font_family = 'Arial',
                  plot_bgcolor = 'white', hovermode = 'x unified',
                  xaxis=dict(showline=True, showgrid=False,
                             showticklabels=True,linecolor='black',
                             linewidth=1, ticks='outside', nticks = 10),
                  yaxis=dict(showline=True, showgrid=False,
                             showticklabels=True, linecolor='black',
                             linewidth=1, ticks='outside',),
                  legend=dict(yanchor='bottom', y=0.8, xanchor='center', x=0.9),
                  margin=dict(l=0, r=0, t=0, b=0))
fig.show()
# fig.write_html('html plots/lmps.html')

In [ ]:
# fig.write_html('html plots/lmps.html')

# 3. Analysis

## 3.1 Power outages
### Did ERCOT counties experience more severe power outages than none ERCOT counties?

Note: `data/eaglei_outages_2021.csv` was too large to add to Github. It can be accessed here: https://figshare.com/articles/dataset/The_Environment_for_Analysis_of_Geoocated_Energy_Information_s_Recorded_Electricity_Outages_2014-2022/24237376

In [ ]:
power_outages = pd.read_csv('data/eaglei_outages_2021.csv', dtype={'fips_code': str})
power_outages = power_outages.loc[power_outages.state.isin(['Texas', 'Oklahoma', 'Louisiana'])] # filter to TX and OK
power_outages['run_start_time'] = pd.to_datetime(power_outages['run_start_time'], format = '%Y-%m-%d %H:%M:%S')
power_outages = (power_outages
                 .loc[(power_outages.run_start_time >= '2021-02-10') & 
                      (power_outages.run_start_time <= '2021-02-20')].reset_index(drop=True))

power_outages.head(3)

In [ ]:
grouping_cols = ['fips_code', 'county', 'state']
dailymax_power = (power_outages
                  .drop(columns='run_start_time')
                  .groupby(by=grouping_cols)
                  .max()
                  .reset_index()
                  .rename(columns={'sum':'max_customers_out'}))

dailymax_power = dailymax_power.merge(df, on = grouping_cols)
dailymax_power['max_pct_out'] = dailymax_power['max_customers_out'] / dailymax_power['population'] * 100

power_df = dailymax_power.copy()
power_df['log_income'] = np.log(power_df['personal_income_thou'])
power_df['log_gdp'] = np.log(power_df['real_gdp'])
power_df['log_pop'] = np.log(power_df['population'])

power_df.head(3)

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf


model = smf.ols(
    formula="max_pct_out ~ C(ercot) + snowfall + log_gdp + log_income + log_pop", 
    data=power_df
).fit()

print(model.summary())


In [ ]:
# Compute mean values for all numeric covariates
covariates = ['snowfall','log_gdp','log_income','log_pop']
calc = power_df.copy()
mean_vals = calc[covariates].mean()

# Create two DataFrames: one for ercot=0, one for ercot=1
df_ercot0 = mean_vals.to_frame().T.assign(ercot=0)
df_ercot1 = mean_vals.to_frame().T.assign(ercot=1)

pred_ercot0 = model.predict(df_ercot0)[0]
pred_ercot1 = model.predict(df_ercot1)[0]

print("Predicted pct_out for Non-ERCOT:", pred_ercot0)
print("Predicted pct_out for ERCOT:", pred_ercot1)

In [ ]:
labels = ["Non-ERCOT", "ERCOT"]
predictions = [pred_ercot0, pred_ercot1]

bar = px.bar(x = labels, y = predictions, text=predictions)
bar.update_layout(height = 500, width = 500, 
                  yaxis_title = 'Predicted % of Outages',
                  xaxis_title = '', font_family='Arial',
                  plot_bgcolor='white')
bar.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
bar.show()

In [ ]:
power_plot = power_df.copy()
power_plot['max_customers_out'] = power_plot['max_customers_out'].replace(np.nan,0)
power_plot['max_pct_out'] = power_plot['max_pct_out'].replace(np.nan,0)
power_plot['max_pct_out'] = [100 if outage > 100 else outage for outage in power_plot['max_pct_out']]

power_plot.head()

In [ ]:
ercot_shape = fips_shapes.loc[fips_shapes.state.isin(['Texas', 'Oklahoma', 'New Mexico', 'Louisiana'])]
ercot_shape = ercot_shape.merge(ercot_labels, on = ['fips_code', 'county', 'state_abbr', 'state'])
ercot_dissolved = ercot_shape.loc[ercot_shape.ercot==1].dissolve().geometry.iloc[0]

if ercot_dissolved.geom_type == 'Polygon':
    x, y = ercot_dissolved.exterior.xy
elif ercot_dissolved.geom_type == 'MultiPolygon':
    # Use the .geoms attribute to iterate through individual polygons
    largest = max(ercot_dissolved.geoms, key=lambda a: a.area)
    lon, lat = largest.exterior.xy

# Convert coordinates to lists
lon = list(lon)
lat = list(lat)

In [ ]:
labels = []

for row in power_plot.max_pct_out:
    if row < 15:
        labels.append('0 - 15%')
    elif row >= 15 and row < 30:
        labels.append('15 - 30%')
    elif row >= 30 and row < 45:
        labels.append('30 - 45%')
    elif row >= 45 and row < 60:
        labels.append('45 - 60%')
    else:
        labels.append('> 60%')

power_plot['label'] = labels

# Map label bins to integers for z
label_to_int = {
    '0 - 15%': 0,
    '15 - 30%': 1,
    '30 - 45%': 2,
    '45 - 60%': 3,
    '> 60%': 4
}
power_plot['z_bin'] = power_plot['label'].map(label_to_int)

# Define colors for the bins
colors = ['#f2f0f7', '#cbc9e2', '#9e9ac8', '#756bb1', '#54278f']

color_labels = list(label_to_int.keys())
zmin = 0
zmax = len(colors) - 1

# Create a discrete colorscale
colorscale = [[i / (len(colors)-1), color] for i, color in enumerate(colors)]

# Plot choropleth using z_bin (hide colorbar)
choropleth = go.Choropleth(
    locations=power_plot['fips_code'],
    geojson=counties,
    z=power_plot['z_bin'],
    zmin=zmin,
    zmax=zmax,
    colorscale=colorscale,
    marker_line_width=0,
    showscale=False,
    hoverinfo='location+z',
    customdata=power_plot[['county', 'max_customers_out', 'max_pct_out']],
    hovertemplate='''<b>%{customdata[0]} County</b><br><br>
                     %{customdata[1]} Customers Out<br>
                     %{customdata[2]:.2f}%<extra></extra>''')

# Create dummy scattergeo traces as discrete legend (off map in Texas)
legend_traces = [
    go.Scattergeo(
        lon=[-121.5], lat=[37 - i * 0.6],  # Space them vertically within TX/OK
        mode='markers',
        marker=dict(size=10, color=color),
        name=label,
        showlegend=True,
        hoverinfo='skip'
    ) for i, (label, color) in enumerate(zip(color_labels, colors))
]

# Add all traces to the figure
fig = go.Figure()
fig.add_trace(choropleth)
for trace in legend_traces:
    fig.add_trace(trace)

fig.add_trace(go.Choropleth(
    geojson=states_geojson,
    locations=['Texas', 'Oklahoma', 'Louisiana'],
    featureidkey="properties.name",
    z=[0, 0, 0],  # Dummy data
    showscale=False,
    marker_line_color="black",
    marker_line_width=1,
    colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],
    hoverinfo='skip'
))

# Add ERCOT boundary
fig.add_trace(go.Scattergeo(
    lon=lon, lat=lat, mode='lines',
    line=dict(width=2, color='brown', dash='dot'),
    name='ERCOT Boundary',  # Legend entry
    showlegend=True, hoverinfo='skip'
))

fig.update_geos(
    visible=False,
    center={"lat": 31.75, "lon": -97.0},
    lonaxis_range=[-107.65, -88],
    lataxis_range=[25.9, 38.2])

fig.update_layout(
    height=400, width=600,
    legend=dict(
        title='Customers Out',
        x=0.75, y=0.975),
    title=dict(
        text='Peak Storm Uri Power Outages',
        x=0, y=0.975),
    margin=dict(t=0, b=0, l=0, r=0),
    dragmode=False, font_family='Arial')

fig.show()
# fig.write_html('html plots/power_outages.html')

## 3.2 FEMA Data
### Hypothesis: ERCOT counties required more FEMA funds than non-ERCOT counties.

In [ ]:
def clean_fips(df, state_col, county_col):
    '''
    converts two int columns of state and county fips codes to a singular str fips code
    '''
    
    fips = []
    for index, row in df.iterrows():
        
        state = str(int(row[state_col]))
        county = str(int(row[county_col]))
        
        if len(county) < 3:
            county = '0' + county
        if len(state) < 2:
            state = '0' + state
            
        fips.append(state+county)
        
    return fips

def clean_fema(fema_df):
    fema_df['declarationDate'] = pd.to_datetime(fema_df['declarationDate'], 
                                                format = '%Y-%m-%dT%H:%M:%S.%fZ')
    
    # filter to October 2020 - April 2021
    fema_df = fema_df.loc[(fema_df.declarationDate >= '2020-10-01') & (fema_df.declarationDate <= '2021-04-30')]

    # add disaster name
    fema_df = fema_df.merge(fema_name_lookup, on = 'disasterNumber')

    # add fips
    fema_df['fips_code'] = clean_fips(fema_df, 'stateNumberCode', 'countyCode')

    return fema_df.reset_index(drop=True)

In [ ]:
tx_fema = pd.read_csv(
    'https://www.fema.gov/api/open/v1/PublicAssistanceFundedProjectsDetails.csv?$filter=state%20eq%20%27Texas%27')

ok_fema = pd.read_csv(
    'https://www.fema.gov/api/open/v1/PublicAssistanceFundedProjectsDetails.csv?$filter=state%20eq%20%27Oklahoma%27')

la_fema = pd.read_csv(
    'https://www.fema.gov/api/open/v1/PublicAssistanceFundedProjectsDetails.csv?$filter=state%20eq%20%27Louisiana%27')

In [ ]:
fema = pd.read_csv('https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries.csv')
fema_name_lookup = fema[['disasterNumber', 'declarationTitle']].drop_duplicates()

In [ ]:
cleaned_tx_fema = clean_fema(tx_fema)
cleaned_ok_fema = clean_fema(ok_fema)
cleaned_la_fema = clean_fema(la_fema)

fema = pd.concat([cleaned_tx_fema, cleaned_ok_fema, cleaned_la_fema])

fema = (
    fema
        .loc[fema.declarationTitle != 'HURRICANE LAURA'][['fips_code', 'projectAmount']]
        .groupby('fips_code')
        .sum()
        .reset_index()
        .merge(df, on = 'fips_code')
)

fema['log_fema'] = np.log(fema['projectAmount'])
fema.head()

In [ ]:
model = smf.ols(
    formula="log_fema ~ C(ercot) + C(state_abbr) + snowfall + log_gdp + log_income + log_pop", 
    data=fema
).fit()

print(model.summary())

In [ ]:
# Set constant values for other covariates
avg_vals = {
    'snowfall': fema['snowfall'].mean(),
    'log_gdp': fema['log_gdp'].mean(),
    'log_income': fema['log_income'].mean(),
    'log_pop': fema['log_pop'].mean(),
}

# Create all 4 group combinations
predict_df = pd.DataFrame([
    {'ercot': 1, 'state_abbr': 'TX', **avg_vals},
    {'ercot': 0, 'state_abbr': 'TX', **avg_vals},
    {'ercot': 0, 'state_abbr': 'LA', **avg_vals},
    {'ercot': 0, 'state_abbr': 'OK', **avg_vals},
])

# Get predictions from your fitted model (e.g., `ols_model`)
predict_df['predicted_log_fema'] = model.predict(predict_df)

predict_df['group'] = predict_df.apply(
    lambda row: f"{'ERCOT' if row.ercot else 'Non-ERCOT'} - {row.state_abbr}", axis=1
)

fig = px.bar(
    predict_df,
    x='group',
    y='predicted_log_fema',
    labels={'group': 'County Group', 'predicted_log_fema': 'Predicted log(FEMA Aid)'},
    title='Predicted FEMA Aid by ERCOT Status and State'
)

fig.update_layout(template='plotly_white', height = 400, width = 700)
fig.show()

## 3.3 Zillow Data

https://www.zillow.com/research/data/

In [ ]:
zillow = pd.read_csv(
    'https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv?t=1744164899'
)
zillow = zillow.melt(id_vars = zillow.columns[:9], 
                     value_vars = zillow.columns[9:],
                     var_name = 'Month', 
                     value_name = 'ZHVI')

zillow = zillow.loc[
    (zillow.Month >= '2020-02-01') & 
    (zillow.Month <= '2022-02-01') &
    (zillow.State.isin(['TX', 'OK', 'LA']))].reset_index(drop=True)

zillow.Month = pd.to_datetime(zillow.Month, format = '%Y-%m-%d')
zillow.RegionName = [county.replace(' County', '') for county in zillow.RegionName]
zillow.RegionName = [county.replace(' Parish', '') for county in zillow.RegionName]
zillow['fips_code'] = clean_fips(zillow, 'StateCodeFIPS', 'MunicipalCodeFIPS')

zillow = zillow[['RegionName', 'State', 'Month', 'ZHVI', 'fips_code']].rename(columns = {'RegionName': 'county', 'State': 'state_abbr'})
zillow['post_uri'] = (zillow['Month'] >= '2021-02-01').astype(int)

zillow = zillow.merge(df, on = ['county', 'state_abbr', 'fips_code'])
zillow['did_interaction'] = zillow['ercot'] * zillow['post_uri']
zillow['log_zhvi'] = np.log(zillow['ZHVI'])
zillow = zillow.reset_index(drop=True)

zillow.head(2)

In [ ]:
# set the panel index
zillow = zillow.set_index(['fips_code', 'Month'])

y, X = patsy.dmatrices(
    'log_zhvi ~ C(state_abbr) + snowfall + log_gdp + log_income + log_pop + ercot + post_uri + did_interaction',
    data=zillow,
    return_type='dataframe'
)

# fit the model with entity fixed effects (county) and clustered SEs
model = PanelOLS(y, X)
results = model.fit(cov_type='clustered', cluster_entity=True)

print(results.summary)

In [ ]:
# Group and prepare the data
avg_trends = zillow.groupby(['Month', 'ercot'])['log_zhvi'].mean().unstack()

# Create the plot
plt.figure(figsize=(9, 4))
plt.plot(avg_trends.index, avg_trends[1], label='ERCOT counties')
plt.plot(avg_trends.index, avg_trends[0], label='Non-ERCOT counties', linestyle='--')

# Add vertical line for Winter Storm Uri
plt.axvline(pd.to_datetime('2021-02-01'), color='red', linestyle=':', label='Winter Storm Uri')

# Labels and formatting
plt.title('Average Log Home Value (ZHVI) Over Time', fontname='Arial')
plt.ylabel('log(ZHVI)', fontname='Arial')
# plt.xlabel('Month', fontname='Arial')
plt.legend()
plt.xticks(fontname='Arial')
plt.yticks(fontname='Arial')
plt.tick_params(labelsize=10)
plt.tight_layout()

# No grid
plt.grid(False)

plt.show()

In [ ]:
zillow['Month'] = pd.to_datetime(zillow['Month']).values.astype('datetime64[M]')
# zillow['Month'] = zillow['Month'].dt.date

In [ ]:
covid['total_dose1'] = covid['total_dose1'].astype('float')
covid['total_complete'] = covid['total_complete'].astype('float')
covid['Month'] = pd.to_datetime(covid.Month)
covid['county'] = covid.county.apply(lambda x:x.replace(' County', ''))


zillow = zillow.merge(covid, on = ['Month', 'fips_code', 'county', 'state_abbr'])
zillow.head(2)

In [ ]:
from linearmodels.panel import PanelOLS

# Set multi-index for panel data (entity and time)
zillow = zillow.set_index(['county', 'Month'])

# Correct regression formula without time-invariant variables
formula = 'ZHVI ~ did_interaction + total_dose1 + total_complete + EntityEffects + TimeEffects'

model = PanelOLS.from_formula(formula, data=zillow)
results = model.fit(cov_type='clustered', cluster_entity=True)

print(results.summary)

## 3.4 Synthetic control method

### Goal

Construct a synthetic ERCOT: a weighted combination of non-ERCOT counties (or other regions) that did have stronger interconnections. This synthetic version mimics ERCOT counties before the storm, and we then compare outcomes after Uri to estimate the causal impact of ERCOT’s grid isolation.

In [ ]:
# try smoothing power outage data by hour instead of 15 minute intervals

hourly_outages = pd.read_csv('data/eaglei_outages_2021.csv', dtype={'fips_code': str}).rename(columns={'sum':'customers_out'})

hourly_outages = hourly_outages.loc[(hourly_outages.state.isin(['Texas', 'Oklahoma', 'Louisiana'])) &
                                    (hourly_outages.run_start_time >= '2021-02-10') & 
                                    (hourly_outages.run_start_time <= '2021-02-20')].reset_index(drop=True)

customers = pd.read_csv('data/power_customers.csv', dtype={'fips_code': str})

hourly_outages = hourly_outages.merge(customers, on = 'fips_code')

hourly_outages.head()

In [ ]:
# find max outages per hour
max_outages = hourly_outages.copy()
max_outages.run_start_time = pd.to_datetime(max_outages.run_start_time)
max_outages['hour'] = max_outages.run_start_time.dt.hour
max_outages['date'] = max_outages.run_start_time.dt.date

max_outages = (max_outages
               .drop(columns='run_start_time')
               .groupby(by=['fips_code', 'county', 'state', 'customers', 'date', 'hour'])
               .max().reset_index()
               .merge(df, on = ['fips_code', 'county', 'state']))

max_outages['datetime'] = pd.to_datetime(max_outages['date']) + pd.to_timedelta(max_outages['hour'], unit='h')

max_outages.head(3)

In [ ]:
# analysis with percent of customers out
max_outages['pct_out'] = max_outages.customers_out / max_outages.customers * 100

# update outliers to 100% outages
max_outages['pct_out'] = [100 if pct > 100 else pct for pct in max_outages['pct_out']]

treated_df = max_outages[max_outages['ercot'] == 1]
control_df = max_outages[max_outages['ercot'] == 0]

# Aggregate ERCOT into a single time series
ercot_series = treated_df.groupby('datetime')['pct_out'].mean().sort_index()
 
donor_matrix = control_df.pivot_table(index='datetime', columns='fips_code', values='pct_out')

# Step 1: Filter columns with enough data
min_required = int(donor_matrix.shape[0] * 0.8)
donor_matrix_filtered = donor_matrix.loc[:, donor_matrix.notna().sum() >= min_required]

donor_matrix_filled = donor_matrix_filtered.ffill(limit=2).bfill(limit=2)
donor_matrix_final = donor_matrix_filled.dropna()

# Align ERCOT series to match donor matrix timestamps
ercot_series_aligned = ercot_series.loc[ercot_series.index.isin(donor_matrix_final.index)]

# Inputs for synthetic control
X = donor_matrix_final.values  # shape: (T, N_donors)
y = ercot_series_aligned.values  # shape: (T,)

In [ ]:
# covariate matching

# Use the earliest timestamp to get one row per county
first_obs = max_outages.sort_values('datetime').drop_duplicates('fips_code')

# Select covariates and standardize
covariates = first_obs.set_index('fips_code')[['snowfall', 'log_pop', 'log_gdp', 'log_income']]
covariates = (covariates - covariates.mean()) / covariates.std()  # Standardize

# Align to donor_matrix columns (same counties)
Z = covariates.loc[donor_matrix_final.columns]  # shape: (N_donors, K)
Z_treated = covariates.loc[max_outages[max_outages['ercot'] == 1]['fips_code'].unique()].mean()  # ERCOT avg

alpha = 0.5  # weight between time series and covariate fit

# Compute covariate weights (inverse variance)
covar_var = Z.var()
covar_weights = 1 / covar_var**1.5

def combined_loss(w):
    y_loss = np.sum((y - X @ w)**2)
    z_synthetic = Z.values.T @ w
    z_diff = (Z_treated.values - z_synthetic)**2
    z_loss = np.sum(z_diff * covar_weights.values)  # weighted covariate loss
    ridge_penalty = 0.001 * np.sum(w**2)
    return alpha * y_loss + (1 - alpha) * z_loss + ridge_penalty

res = minimize(
    combined_loss,
    x0=np.ones(X.shape[1]) / X.shape[1],
    method='trust-constr',
    constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
    bounds=[(0, 1)] * X.shape[1],
    options={'disp': True}
)

weights = res.x
synthetic_series = X @ weights

In [ ]:
# Plot synthetic control result
fig = go.Figure()
fig.add_trace(go.Scatter(x=ercot_series_aligned.index, y=ercot_series_aligned,
                         mode='lines', name='Actual ERCOT', 
                         hovertemplate = '%{y:.2f}% Out of Power<extra></extra>'))
fig.add_trace(go.Scatter(x=ercot_series_aligned.index, y=synthetic_series,
                         mode='lines', name='Synthetic ERCOT',
                         hovertemplate = '%{y:.2f}% Out of Power<extra></extra>'))

fig.update_layout(
    title='Synthetic Control with Covariate Matching: ERCOT Outages During Uri',
    yaxis_title='Percentage of Customers Without Power',
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=40, r=40, t=60, b=40),
    height=400, width = 600,
    font_family='Arial',
    plot_bgcolor = 'white',
    hovermode = 'x unified'
)

fig.show()
# fig.write_html('html plots/synthetic_control.html')

In [ ]:
# define pre/post period for Winter Storm Uri
pre_period = ercot_series_aligned.index < '2021-02-12'
post_period = ercot_series_aligned.index >= '2021-02-12'

# run placebo SCM for each donor as if it were "treated"
placebo_gaps = []
rmspe_ratios = []

for placebo_col in donor_matrix_final.columns:
    y_placebo = donor_matrix_final[placebo_col].values
    X_placebo = donor_matrix_final.drop(columns=placebo_col).values

    if X_placebo.shape[1] < 2:
        continue

    def loss_placebo(w):
        return np.sum((y_placebo - X_placebo @ w)**2)

    res_p = minimize(loss_placebo, x0=np.ones(X_placebo.shape[1]) / X_placebo.shape[1],
                     constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
                     bounds=[(0, 1)] * X_placebo.shape[1])

    if not res_p.success:
        continue

    w_p = res_p.x
    synthetic_placebo = X_placebo @ w_p
    gap = y_placebo - synthetic_placebo

    rmspe_pre = np.sqrt(np.mean(gap[pre_period]**2))
    rmspe_post = np.sqrt(np.mean(gap[post_period]**2))

    if rmspe_pre > 0:
        rmspe_ratios.append(rmspe_post / rmspe_pre)
        placebo_gaps.append(gap)

# calculate ERCOT RMSPE ratio
ercot_gap = ercot_series_aligned.values - synthetic_series
ercot_rmspe_pre = np.sqrt(np.mean(ercot_gap[pre_period]**2))
ercot_rmspe_post = np.sqrt(np.mean(ercot_gap[post_period]**2))
ercot_rmspe_ratio = ercot_rmspe_post / ercot_rmspe_pre

# compute p-value
p_val = np.mean([r >= ercot_rmspe_ratio for r in rmspe_ratios])

fig_spaghetti = go.Figure()

# add placebo gaps
for gap in placebo_gaps:
    fig_spaghetti.add_trace(go.Scatter(x=ercot_series_aligned.index, y=gap,
                                       mode='lines', line=dict(color='gray', width=1),
                                       opacity=0.3, showlegend=False, 
                                       hovertemplate = '%{y:.2f}% Gap<extra></extra>'))

# add ERCOT gap
fig_spaghetti.add_trace(go.Scatter(x=ercot_series_aligned.index, y=ercot_gap,
                                   mode='lines', name='ERCOT Gap',
                                   line=dict(width=2), hovertemplate = '%{y:.2f}% Gap<extra></extra>'))

# add vertical line for storm start
fig_spaghetti.add_vline(x=pd.Timestamp('2021-02-12'), line_width=2, line_dash='dash', line_color='red')

fig_spaghetti.update_layout(
    title=f'Placebo Gaps vs. ERCOT | RMSPE Ratio: {ercot_rmspe_ratio:.2f}, p = {p_val:.2f}',
    yaxis_title='Gap (Actual - Synthetic % Out)',
    margin=dict(l=40, r=40, t=60, b=40),
    font_family='Arial', plot_bgcolor = 'white',
    height = 400, width = 600,
    legend=dict(x=0.85, y=0.99)
)

# fig_spaghetti.write_html('html plots/spaghetti_plot.html')
fig_spaghetti.show()